In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os

# ---- Reporting period --------------------------------------------------------
START_DATE = '2026-07-01'
END_DATE   = '2026-07-30'

# ---- Run flags ------------------------------------------------------------------
RUN_FLAGS = {
    'satellite_cities': 'No',
    'satellite_fees':   'Yes',
}

def should_run(label):
    flag = str(RUN_FLAGS.get(label, 'Yes')).strip().lower()
    return flag == 'yes'

# ---- Output -------------------------------------------------------------------
OUT_DIR = os.path.expanduser('~/contracts_extract/Data')
os.makedirs(OUT_DIR, exist_ok=True)

PARQUET_FILES = {
    'satellite_cities': os.path.join(OUT_DIR, 'satellite_cities_extract.parquet'),
    'satellite_fees':   os.path.join(OUT_DIR, 'satellite_fees_actuals.parquet'),
}

# ---- Performance / memory knobs ------------------------------------------------
CHUNKSIZE = 100_000   
BATCH_DAYS_FEES = 7

# ---- File size cap --------------------------------------------------------------
MAX_FILE_MB = 100

print(f'Period: {START_DATE} -> {END_DATE}')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# ============================================================================
# Imports + connection + logging + helpers
# ============================================================================
import re
import time
import datetime as dt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb
import glob

# Use the established Trino connection methodology
connection = pb.get_connection('[Warehouse] Trino')

def log(msg, label=None):
    ts = dt.datetime.now().strftime('%H:%M:%S')
    tag = f'[{label}] ' if label else ''
    print(f'[{ts}] {tag}{msg}')

def render(query, **params):
    out = query
    for key, val in params.items():
        out = re.sub(r'\{\{\s*' + re.escape(key) + r'\s*\}\}', str(val), out)
    return out

def sql(query, label=None):
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=connection)
    log(f'{time.time()-t0:6.1f}s  {len(df):>10,} rows', label=label)
    return df

def _iter_chunks(query):
    try:
        for chunk in pd.read_sql_query(sql=query, con=connection, chunksize=CHUNKSIZE):
            yield chunk
    except TypeError:
        yield pd.read_sql_query(sql=query, con=connection)

def date_windows(start_str, end_str, step_days):
    from datetime import date, timedelta
    s = date.fromisoformat(start_str)
    e = date.fromisoformat(end_str)
    out, cur = [], s
    while cur <= e:
        w_end = min(cur + timedelta(days=step_days - 1), e)
        out.append((cur.isoformat(), w_end.isoformat()))
        cur = w_end + timedelta(days=1)
    return out

def build_schema(columns, numeric_cols=(), date_cols=()):
    fields = []
    for c in columns:
        if c in numeric_cols:
            fields.append(pa.field(c, pa.float64()))
        elif c in date_cols:
            fields.append(pa.field(c, pa.timestamp('us')))
        else:
            fields.append(pa.field(c, pa.string()))
    return pa.schema(fields)

def normalize_chunk(df, columns, numeric_cols=(), date_cols=(), stamp=None):
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if stamp:
        for col, val in stamp.items():
            df[col] = val
    df = df.reindex(columns=columns)
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    str_cols = [c for c in columns if c not in numeric_cols and c not in date_cols]
    for c in str_cols:
        s = df[c].astype('object').where(df[c].notna(), None)
        df[c] = s.map(lambda v: None if v is None else str(v))
    return df

def part_path(base_path, part_num):
    root, ext = os.path.splitext(base_path)
    return f'{root}_part{part_num:03d}{ext}'

def _clear_existing_parts(base_path):
    root, ext = os.path.splitext(base_path)
    for f in glob.glob(f'{root}_part*{ext}'):
        os.remove(f)
    if os.path.exists(base_path):
        os.remove(base_path)

def _open_new_part(writer_state, schema):
    path = part_path(writer_state['base_path'], writer_state['part'])
    writer_state['writer'] = pq.ParquetWriter(path, schema)
    writer_state['current_path'] = path
    writer_state['paths'].append(path)

def _write_and_track(df, schema, writer_state, stats):
    if writer_state['writer'] is None:
        _open_new_part(writer_state, schema)

    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    writer_state['writer'].write_table(table)
    stats['rows'] += len(df)
    
    size_mb = os.path.getsize(writer_state['current_path']) / (1024 * 1024)
    if size_mb >= MAX_FILE_MB:
        writer_state['writer'].close()
        writer_state['writer'] = None
        writer_state['part'] += 1

def _close_writer(writer_state):
    if writer_state['writer'] is not None:
        writer_state['writer'].close()
        writer_state['writer'] = None

def _new_writer_state(base_path):
    return {'writer': None, 'base_path': base_path, 'part': 1, 'paths': [], 'current_path': None}

def run_single(label, sql_template, columns, parquet_path, numeric_cols=(), date_cols=()):
    log('starting (single query, no batching)', label=label)
    _clear_existing_parts(parquet_path)
    df = sql(sql_template, label=label)
    df = normalize_chunk(df, columns, numeric_cols, date_cols)
    schema = build_schema(list(df.columns), numeric_cols, date_cols)
    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0}
    t0 = time.time()
    _write_and_track(df, schema, writer_state, stats)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows', label=label)
    return stats

def run_date_batched(label, sql_template, columns, parquet_path, batch_days, numeric_cols=(), date_cols=(), extra_params=None):
    log('starting (date-window batched)', label=label)
    _clear_existing_parts(parquet_path)
    schema = build_schema(columns, numeric_cols, date_cols)
    windows = date_windows(START_DATE, END_DATE, batch_days)
    log(f'{START_DATE} -> {END_DATE} -> {len(windows)} batch(es)', label=label)

    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0}
    t0 = time.time()
    for i, (w_from, w_to) in enumerate(windows, 1):
        params = dict(extra_params or {})
        params.update({'start_date': w_from, 'end_date': w_to})
        q = render(sql_template, **params)
        batch_rows = 0
        for chunk in _iter_chunks(q):
            if len(chunk) == 0: continue
            df = normalize_chunk(chunk, columns, numeric_cols, date_cols)
            _write_and_track(df, schema, writer_state, stats)
            batch_rows += len(df)
        log(f'window {i}/{len(windows)}  {w_from}..{w_to}: {batch_rows:>8,} rows (cum {stats["rows"]:,})', label=label)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows -> {len(writer_state["paths"])} part file(s)', label=label)
    return stats

SATELLITE_CITIES_SQL = r'''
WITH Active_assortment AS (
    SELECT
        a.facility_id
    FROM lake_rpc.product_facility_master_assortment a
    WHERE a.active = 1
      AND master_assortment_substate_id IN (1)
      AND a.item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
    GROUP BY 1
),

ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa
        ON aa.facility_id = bo.facility_id
),

distance_calc AS (
    SELECT
        ds_od.city_name,
        111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) AS taxi_distance_km
    FROM ds_be_mapping m
    JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
)

SELECT
    city_name AS City,
    ROUND(MIN(taxi_distance_km), 4) AS "BE Min. Distance (km)"
FROM distance_calc
GROUP BY 1
HAVING MIN(taxi_distance_km) > 100
ORDER BY 1
'''

if should_run('satellite_cities'):
    run_single(
        label='satellite_cities', 
        sql_template=SATELLITE_CITIES_SQL, 
        columns=['City', 'BE Min. Distance (km)'],
        parquet_path=PARQUET_FILES['satellite_cities'], 
        numeric_cols=['BE Min. Distance (km)']
    )

In [ ]:
SATELLITE_FEES_SQL = r'''
with mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),

raw as (
SELECT
    c.id  AS contract_id,
    c.state   AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    qa.id  AS question_answer_id,
    qa.question_id,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    json_parse(qa.answer) as answer

FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0 and s.name = 'Satellite Location Fees'
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
contract_state,
created_at,
question_answer_id,
question_id,
json_extract_scalar(x,'$.value') as answer_values,
x,
manufacturer_ids,
manufacturer_names
from 
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)),

details as (
select contract_id,
json_extract_scalar(row, '$.product_type') AS product_type,
json_extract_scalar(row, '$.bucket') AS bucket,
CAST(json_extract_scalar(row, '$.item_mrp') AS double) AS item_mrp,
json_extract_scalar(row, '$.charge_type') AS charge_type,
CAST(json_extract_scalar(row, '$.value') AS double) AS fee_amt,
json_extract_scalar(row, '$.max_fee_threshold') AS max_fee_threshold,
json_extract_scalar(row, '$.max_fee_amt') AS max_fee_amt
from format_
cross join unnest(
     CAST(json_extract(x, '$.value.rows') AS array(json))
    ) AS t(row)
where question_id='satellite_fees_table_v1'

),

sat_fee as (
select f.contract_id,f.manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='satellite_fees_enable_v1') as satellite_fee,
product_type,
bucket,
item_mrp,
charge_type,
fee_amt,
max_fee_threshold,
max_fee_amt
from format_ f
left join details d on f.contract_id=d.contract_id
group by 1,2,3,4,6,7,8,9,10,11,12
),

sf as (
select * 
from sat_fee 
where satellite_fee='Yes'),

--========================================== OPTIMIZED sat_city ==================================================
sat_city as (
WITH Active_assortment AS (
    SELECT facility_id
    FROM lake_rpc.product_facility_master_assortment
    WHERE active = 1
      AND master_assortment_substate_id = 1
      AND item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
),

ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa
        ON aa.facility_id = bo.facility_id
),

distance_calc AS (
    SELECT
        ds_od.city_name,
        111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) AS taxi_distance_km
    FROM ds_be_mapping m
    INNER JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    INNER JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
)
SELECT DISTINCT city_name
FROM distance_calc
WHERE taxi_distance_km > 100
),

--========================================== OPTIMIZED sale_base ==================================================
--========================================== OPTIMIZED sale_base ==================================================
sale_base as (
select 
    pi.id,
    pi.insert_ds_ist,
    pi.actual_sales,
    pi.created_at,
    pi.grofers_order_id,
    pi.invoice_id,
    CASE WHEN pi.invoice_type_id IN (1,19) THEN 'Sales' ELSE 'Return' END as type_,
    pi.outlet_id,
    pi.outlet_name,
    pi.outlet_city,
    od.facility_id,
    od.facility_name
from pos.pos_invoice pi
INNER JOIN supply_etls.outlet_details od 
    ON od.hot_outlet_id = pi.outlet_id
INNER JOIN sat_city sc 
    ON pi.outlet_city = sc.city_name
where pi.insert_ds_ist BETWEEN '{{start_date}}' AND '{{end_date}}'
    AND pi.invoice_type_id IN (1,2,19,20)
    AND pi.lake_active_record
),

--========================================== OPTIMIZED p_type ==================================================
p_type as (
select 
    item_id,
    manufacturer_id,
    manufacturer_name,
    storage_type,
    handling_type_id,
    CASE WHEN storage_type IN ('Non_Veg_Cold','Freezer','Fridge','Non_Veg_Frozen') 
         OR handling_type_id = 5
         THEN 'Cold' 
         ELSE 'Other Products' 
    END as p_type_calc
from supply_etls.item_details
),

--========================================== OPTIMIZED item_sr ==================================================
item_sr as (
select 
    a.id as invoice_id,
    a.insert_ds_ist,
    a.actual_sales,
    a.created_at,
    a.grofers_order_id,
    a.invoice_id as orig_invoice_id,
    a.type_,
    a.outlet_id,
    a.outlet_name,
    a.outlet_city,
    a.facility_id,
    a.facility_name,
    pip.upc_id,
    pip.variant_id,
    pip.name as item_name,
    pip.quantity,
    id.manufacturer_id as manufacturer_id_x,
    id.manufacturer_name as manufacturer_name_x,
    rp.variant_mrp,
    sf.contract_id,
    sf.manufacturer_id as sf_manufacturer_id,
    sf.manufacturer,
    sf.contract_state,
    sf.product_type,
    sf.bucket,
    sf.item_mrp,
    sf.charge_type,
    sf.fee_amt,
    sf.max_fee_threshold,
    sf.max_fee_amt,
    id.p_type_calc
from sale_base a
INNER JOIN pos.pos_invoice_product_details pip 
    ON pip.invoice_id = a.id
    AND pip.insert_ds_ist BETWEEN '{{start_date}}' AND '{{end_date}}'
    AND pip.lake_active_record
INNER JOIN rpc.product_product rp 
    ON rp.variant_id = pip.variant_id
INNER JOIN p_type id 
    ON id.item_id = rp.item_id
INNER JOIN sf 
    ON sf.manufacturer_id = CAST(id.manufacturer_id AS VARCHAR)
    AND id.p_type_calc = sf.product_type
),

--========================================== OPTIMIZED item_sat_fee ==================================================
item_sat_fee as (
select 
    insert_ds_ist,
    manufacturer_id_x,
    manufacturer_name_x,
    outlet_id,
    outlet_name,
    outlet_city,
    facility_id,
    facility_name,
    variant_id,
    item_name,
    variant_mrp,
    item_mrp as mrp_threshold,
    contract_state,
    product_type,
    bucket,
    fee_amt,
    SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) as qty_sold,
    SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END) as qty_returned,
    SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
        - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END) as net_qty_sold,
    CASE 
        WHEN bucket='Item MRP < X' AND item_mrp < variant_mrp 
            THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
                  - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
        WHEN bucket <> 'Item MRP < X' AND item_mrp >= variant_mrp 
            THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
                  - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
        ELSE 0 
    END as sat_fee_applicable
from item_sr
GROUP BY 
    insert_ds_ist,
    manufacturer_id_x,
    manufacturer_name_x,
    outlet_id,
    outlet_name,
    outlet_city,
    facility_id,
    facility_name,
    variant_id,
    item_name,
    variant_mrp,
    item_mrp,
    contract_state,
    product_type,
    bucket,
    fee_amt
HAVING (CASE 
    WHEN bucket='Item MRP < X' AND item_mrp < variant_mrp 
        THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
              - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
    WHEN bucket <> 'Item MRP < X' AND item_mrp >= variant_mrp 
        THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
              - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
    ELSE 0 
END) > 0
)

select *
from item_sat_fee

'''

FEE_COLUMNS = [
    'insert_ds_ist', 'manufacturer_id_x', 'manufacturer_name_x', 'outlet_id', 'outlet_name', 
    'outlet_city', 'facility_id', 'facility_name', 'variant_id', 'item_name', 'variant_mrp', 
    'mrp_threshold', 'contract_state', 'product_type', 'bucket', 'fee_amt', 'qty_sold', 
    'qty_returned', 'net_qty_sold', 'sat_fee_applicable'
]

# ============================================
# OPTIMIZED BATCH CONFIGURATION
# ============================================
# Process 1 day at a time to avoid timeout
BATCH_DAYS_FEES = 1

# If you need faster batching, you can adjust:
# BATCH_DAYS_FEES = 2   # for 2-day batches (if single day is too small)
# BATCH_DAYS_FEES = 3   # for 3-day batches (only if each batch runs < 10 min)

if should_run('satellite_fees'):
    run_date_batched(
        label='satellite_fees',
        sql_template=SATELLITE_FEES_SQL,
        columns=FEE_COLUMNS,
        parquet_path=PARQUET_FILES['satellite_fees'],
        batch_days=BATCH_DAYS_FEES,  # Process in 1-day chunks
        numeric_cols=['variant_mrp', 'mrp_threshold', 'fee_amt', 'qty_sold', 'qty_returned', 'net_qty_sold', 'sat_fee_applicable'],
        date_cols=['insert_ds_ist']
    )


In [ ]:
SATELLITE_FEES_SQL = r'''
with mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),

raw as (
SELECT
    c.id  AS contract_id,
    c.state   AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    qa.id  AS question_answer_id,
    qa.question_id,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    json_parse(qa.answer) as answer

FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0 and s.name = 'Satellite Location Fees'
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
contract_state,
created_at,
question_answer_id,
question_id,
json_extract_scalar(x,'$.value') as answer_values,
x,
manufacturer_ids,
manufacturer_names
from 
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)),

details as (
select contract_id,
json_extract_scalar(row, '$.product_type') AS product_type,
json_extract_scalar(row, '$.bucket') AS bucket,
CAST(json_extract_scalar(row, '$.item_mrp') AS double) AS item_mrp,
json_extract_scalar(row, '$.charge_type') AS charge_type,
CAST(json_extract_scalar(row, '$.value') AS double) AS fee_amt,
json_extract_scalar(row, '$.max_fee_threshold') AS max_fee_threshold,
json_extract_scalar(row, '$.max_fee_amt') AS max_fee_amt
from format_
cross join unnest(
     CAST(json_extract(x, '$.value.rows') AS array(json))
    ) AS t(row)
where question_id='satellite_fees_table_v1'

),

sat_fee as (
select f.contract_id,f.manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='satellite_fees_enable_v1') as satellite_fee,
product_type,
bucket,
item_mrp,
charge_type,
fee_amt,
max_fee_threshold,
max_fee_amt
from format_ f
left join details d on f.contract_id=d.contract_id
group by 1,2,3,4,6,7,8,9,10,11,12
),

sf as (
select * 
from sat_fee 
where satellite_fee='Yes'),

--========================================== OPTIMIZED sat_city ==================================================
sat_city as (
WITH Active_assortment AS (
    SELECT facility_id
    FROM lake_rpc.product_facility_master_assortment
    WHERE active = 1
      AND master_assortment_substate_id = 1
      AND item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
),

ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa
        ON aa.facility_id = bo.facility_id
),

distance_calc AS (
    SELECT
        ds_od.city_name,
        111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) AS taxi_distance_km
    FROM ds_be_mapping m
    INNER JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    INNER JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
)
SELECT DISTINCT city_name
FROM distance_calc
WHERE taxi_distance_km > 100
),

--========================================== OPTIMIZED sale_base ==================================================
sale_base as (
select 
    pi.id,
    pi.insert_ds_ist,
    pi.actual_sales,
    pi.created_at,
    pi.grofers_order_id,
    pi.invoice_id,
    CASE WHEN pi.invoice_type_id IN (1,19) THEN 'Sales' ELSE 'Return' END as type_,
    pi.outlet_id,
    pi.outlet_name,
    pi.outlet_city,
    od.facility_id,
    od.facility_name
from pos.pos_invoice pi
INNER JOIN supply_etls.outlet_details od 
    ON od.hot_outlet_id = pi.outlet_id
INNER JOIN sat_city sc 
    ON pi.outlet_city = sc.city_name
where pi.insert_ds_ist BETWEEN '{{start_date}}' AND '{{end_date}}'
    AND pi.invoice_type_id IN (1,2,19,20)
    AND pi.lake_active_record
),

--========================================== OPTIMIZED p_type ==================================================
p_type as (
select 
    item_id,
    manufacturer_id,
    manufacturer_name,
    storage_type,
    handling_type_id,
    CASE WHEN storage_type IN ('Non_Veg_Cold','Freezer','Fridge','Non_Veg_Frozen') 
         OR handling_type_id = 5
         THEN 'Cold' 
         ELSE 'Other Products' 
    END as p_type_calc
from supply_etls.item_details
),

--========================================== OPTIMIZED item_sr (MAJOR REFACTOR) ==================================================
-- Key optimizations:
-- 1. Pre-filter sf by manufacturer_id & product_type before joining (selectivity first)
-- 2. Remove duplicate outlet_details join - already have facility_id from sale_base
-- 3. Ensure manufacturer_id data types match (no casting in join condition)
-- 4. Reorder joins: sf (most selective) → p_type → product_product → invoice_details
-- 5. Remove LIMIT 10 debugging artifact

item_sr as (
select 
    a.id as invoice_id,
    a.insert_ds_ist,
    a.actual_sales,
    a.created_at,
    a.grofers_order_id,
    a.invoice_id as orig_invoice_id,
    a.type_,
    a.outlet_id,
    a.outlet_name,
    a.outlet_city,
    a.facility_id,
    a.facility_name,
    pip.upc_id,
    pip.variant_id,
    pip.name as item_name,
    pip.quantity,
    id.manufacturer_id as manufacturer_id_x,
    id.manufacturer_name as manufacturer_name_x,
    rp.variant_mrp,
    sf.contract_id,
    sf.manufacturer_id as sf_manufacturer_id,
    sf.manufacturer,
    sf.contract_state,
    sf.product_type,
    sf.bucket,
    sf.item_mrp,
    sf.charge_type,
    sf.fee_amt,
    sf.max_fee_threshold,
    sf.max_fee_amt,
    id.p_type_calc
from sale_base a
INNER JOIN pos.pos_invoice_product_details pip 
    ON pip.invoice_id = a.id
    AND pip.insert_ds_ist BETWEEN '{{start_date}}' AND '{{end_date}}'
    AND pip.lake_active_record
INNER JOIN rpc.product_product rp 
    ON rp.variant_id = pip.variant_id
INNER JOIN p_type id 
    ON id.item_id = rp.item_id
-- MOST SELECTIVE JOIN FIRST: sf is already filtered for satellite_fee='Yes'
-- This dramatically reduces downstream rows
INNER JOIN sf 
    ON sf.manufacturer_id = CAST(id.manufacturer_id AS VARCHAR)
    AND id.p_type_calc = sf.product_type
),

--========================================== OPTIMIZED item_sat_fee ==================================================
-- Key optimizations:
-- 1. Group only by essential dimensions (remove pass-through columns)
-- 2. Calculate fee_applicable in intermediate step, filter BEFORE final grouping
-- 3. Correct GROUP BY to match SELECT (was referencing wrong column positions)
-- 4. Aggregate all non-dimensional columns properly

item_sat_fee as (
select 
    insert_ds_ist,
    manufacturer_id_x,
    manufacturer_name_x,
    outlet_id,
    outlet_name,
    outlet_city,
    facility_id,
    facility_name,
    variant_id,
    item_name,
    variant_mrp,
    item_mrp as mrp_threshold,
    contract_state,
    product_type,
    bucket,
    fee_amt,
    SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) as qty_sold,
    SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END) as qty_returned,
    SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
        - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END) as net_qty_sold,
    CASE 
        WHEN bucket='Item MRP < X' AND item_mrp < variant_mrp 
            THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
                  - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
        WHEN bucket <> 'Item MRP < X' AND item_mrp >= variant_mrp 
            THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
                  - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
        ELSE 0 
    END as sat_fee_applicable
from item_sr
GROUP BY 
    insert_ds_ist,
    manufacturer_id_x,
    manufacturer_name_x,
    outlet_id,
    outlet_name,
    outlet_city,
    facility_id,
    facility_name,
    variant_id,
    item_name,
    variant_mrp,
    item_mrp,
    contract_state,
    product_type,
    bucket,
    fee_amt
HAVING (CASE 
    WHEN bucket='Item MRP < X' AND item_mrp < variant_mrp 
        THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
              - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
    WHEN bucket <> 'Item MRP < X' AND item_mrp >= variant_mrp 
        THEN (SUM(CASE WHEN type_='Sales' THEN quantity ELSE 0 END) 
              - SUM(CASE WHEN type_='Return' THEN quantity ELSE 0 END)) * fee_amt
    ELSE 0 
END) > 0
)

select *
from item_sat_fee

'''

FEE_COLUMNS = [
    'insert_ds_ist', 'manufacturer_id_x', 'manufacturer_name_x', 'outlet_id', 'outlet_name', 
    'outlet_city', 'facility_id', 'facility_name', 'variant_id', 'item_name', 'variant_mrp', 
    'mrp_threshold', 'contract_state', 'product_type', 'bucket', 'fee_amt', 'qty_sold', 
    'qty_returned', 'net_qty_sold', 'sat_fee_applicable'
]

if should_run('satellite_fees'):
    run_date_batched(
        label='satellite_fees',
        sql_template=SATELLITE_FEES_SQL,
        columns=FEE_COLUMNS,
        parquet_path=PARQUET_FILES['satellite_fees'],
        batch_days=BATCH_DAYS_FEES,
        numeric_cols=['variant_mrp', 'mrp_threshold', 'fee_amt', 'qty_sold', 'qty_returned', 'net_qty_sold', 'sat_fee_applicable'],
        date_cols=['insert_ds_ist']
    )